# Bharat-EO-Bench · Prithvi-Seg **v3 ABLATION** — what each improvisation is worth

Runs the proven v2 model (UPerNet-on-ViT + Lovász + LLRD + TTA → 0.663) on the
**Dataset v2** chips (`bharat-eo-chips-dsv2`), consuming everything the richer data adds:
- **17 input bands**: Prithvi's native 6 optical feed the ViT; the extra 11 (red-edge
  B5/B6/B7, SAR VV/VH, elevation/slope/aspect, rain/temp/nightlights) feed an **aux stem**
  fused as a 5th decoder pyramid level.
- **Confidence-masking** (`USE_CONF`): ignore pixels where <2 label sources agree.
- **Soft labels** (`USE_SOFT`): distil from Dynamic World class probabilities (aux `dwp*`).
- **Copy-paste rare-class aug** (`USE_COPYPASTE`): paste snow/wetland/mangrove/moss instances.
- **Class-balanced sampler** (`USE_BALANCED`): resample by dominant class.
Every improvisation is an ablation flag → measure each contribution.

Kaggle: + Add Input `bharat-eo-chips-dsv2`, GPU T4, Internet ON, Run All. ~2–3 h.


In [1]:
import importlib.metadata as md
KEEP = {p: md.version(p) for p in ("numpy","pandas","scipy","scikit-learn")}
!pip install -q segmentation-models-pytorch timm
!pip install -q terratorch
pins=" ".join(f"{k}=={v}" for k,v in KEEP.items())
!pip install -q --no-deps --force-reinstall {pins}
!python -c "import numpy;print('numpy',numpy.__version__)"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 643.8/643.8 kB 16.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.8/136.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 83.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os, glob, random, json, gc, time, numpy as np, pandas as pd, torch, torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, jaccard_score, accuracy_score
dev="cuda" if torch.cuda.is_available() else "cpu"; print("device",dev)
K, EPOCHS, WARMUP, SEED, MODE = 11, 35, 3, 0, "finetune"      # MODE: 'finetune' | 'frozen'
# dsv2 img band order: B2 B3 B4 B5 B6 B7 B8A B11 B12 VV VH elev slope aspect rain temp ntl
PRITHVI_IDX = [0,1,2,6,7,8]                                   # Prithvi native 6 optical
AUX_IDX     = [3,4,5,9,10,11,12,13,14,15,16]                  # the 11 extra context bands
DW_TO_IDX   = [7,0,2,8,3,1,4,5,6]                             # Dynamic World class i -> our class
RARE_CP     = [6,8,9,10]                                      # snow, wetland, mangrove, moss
# --- ablation flags (turn each improvisation on/off) ---
USE_AUXSTEM, USE_CONF, USE_SOFT, USE_COPYPASTE, USE_BALANCED = True, True, True, True, True
LAM_SOFT, CONF_MIN, P_CP = 0.3, 2, 0.4


device cuda


In [3]:
# --- locate dsv2 chips (img[17] + aux[14] + lbl) -----------------------------
import zipfile
def _find():
    for pat in ("/kaggle/input/**/img/*.npy","/kaggle/working/**/img/*.npy","data/chips_dsv2/img/*.npy"):
        h=glob.glob(pat,recursive=True)
        if h: return h
    return []
imgs=_find()
if not imgs:
    for z in glob.glob("/kaggle/input/**/*.zip",recursive=True):
        try:
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith(".npy") for n in zf.namelist()[:3000]): zf.extractall("/kaggle/working/chips")
        except Exception: pass
    imgs=_find()
if not imgs:
    print("!! No dsv2 chips. Attach 'bharat-eo-chips-dsv2'."); raise SystemExit
ROOT=os.path.dirname(os.path.dirname(imgs[0])); man=pd.read_csv(os.path.join(ROOT,"manifest.csv"))
if "split" not in man.columns: man["split"]="train"
print("root",ROOT,"| n=",len(imgs),"| splits",man.split.value_counts().to_dict())
def load_img(c): return np.transpose(np.load(f"{ROOT}/img/{c}.npy").astype(np.float32),(2,0,1))       # [17,H,W]
def load_aux(c): return np.transpose(np.load(f"{ROOT}/aux/{c}.npy").astype(np.float32),(2,0,1))       # [14,H,W]
def load_lbl(c): return np.load(f"{ROOT}/lbl/{c}.npy").astype(np.uint8)
CIN=len(load_img(man.chip.iloc[0])); print("input channels:",CIN,"| Prithvi",len(PRITHVI_IDX),"| aux",len(AUX_IDX))
tr=man.loc[man.split=="train","chip"].tolist(); va=man.loc[man.split.isin(["val","test"]),"chip"].tolist()
st=np.stack([load_img(c) for c in tr[:200]]); MU=st.mean((0,2,3),keepdims=True)[0]; SD=st.std((0,2,3),keepdims=True)[0]+1e-6
cnt=np.zeros(K)
for c in tr:
    v=load_lbl(c).ravel(); v=v[v!=255]; cnt+=np.bincount(v,minlength=K)
CW=torch.tensor(np.clip(cnt.sum()/(K*np.clip(cnt,1,None)),0.5,8.0).astype(np.float32))
CP_POOL=man.loc[man.target.isin(RARE_CP),"chip"].tolist() if "target" in man.columns else []
print("copy-paste rare pool:",len(CP_POOL),"chips | class pixel counts:",dict(enumerate(cnt.astype(int))))

def soft_from_aux(aux):                                       # aux[2:11]=dwp0..8 -> 11-class soft target
    dwp=aux[2:11]; soft=np.zeros((K,)+dwp.shape[1:],np.float32)
    for i,c in enumerate(DW_TO_IDX): soft[c]+=dwp[i]
    return soft/(soft.sum(0,keepdims=True)+1e-6)

class DS(torch.utils.data.Dataset):
    def __init__(s,ch,tr=False): s.ch,s.tr=ch,tr
    def __len__(s): return len(s.ch)
    def __getitem__(s,i):
        c=s.ch[i]; raw=load_img(c); x=(raw-MU)/SD; aux=load_aux(c); y=load_lbl(c).astype(np.int64)
        soft=soft_from_aux(aux) if USE_SOFT else np.zeros((K,)+y.shape,np.float32)
        if USE_CONF: y=np.where(aux[12]>=CONF_MIN, y, 255).astype(np.int64)   # low-agreement -> ignore
        if s.tr and USE_COPYPASTE and CP_POOL and random.random()<P_CP:
            sc=random.choice(CP_POOL); sx=(load_img(sc)-MU)/SD; sl=load_lbl(sc)
            m=np.isin(sl,RARE_CP)
            if m.any():
                x[:,m]=sx[:,m]; y[m]=sl[m]; soft[:,m]=0                       # pasted -> hard label only
        if s.tr:
            if random.random()<.5: x=x[:,:,::-1]; y=y[:,::-1]; soft=soft[:,:,::-1]
            if random.random()<.5: x=x[:,::-1,:]; y=y[::-1,:]; soft=soft[:,::-1,:]
            k=random.randint(0,3); x=np.rot90(x,k,(1,2)); y=np.rot90(y,k); soft=np.rot90(soft,k,(1,2))
        return (torch.from_numpy(x.copy()).float(), torch.from_numpy(soft.copy()).float(),
                torch.from_numpy(y.copy()).long())

def make_sampler():
    if not USE_BALANCED or "dominant" not in man.columns: return None
    dom=man.set_index("chip").loc[tr,"dominant"].to_numpy()
    freq=np.bincount(dom[dom>=0],minlength=K).astype(np.float32); freq[freq==0]=1
    w=np.array([1.0/freq[d] if d>=0 else 1.0/freq.mean() for d in dom])
    return torch.utils.data.WeightedRandomSampler(torch.from_numpy(w).double(),len(w),replacement=True)


root /kaggle/input/datasets/crakashpreetham/bharat-eo-chips-dsv2 | n= 942 | splits {'train': 674, 'test': 134, 'val': 134}
input channels: 17 | Prithvi 6 | aux 11
copy-paste rare pool: 88 chips | class pixel counts: {0: np.int64(10429265), 1: np.int64(1381690), 2: np.int64(3688891), 3: np.int64(12206252), 4: np.int64(1053751), 5: np.int64(2705815), 6: np.int64(560275), 7: np.int64(1004740), 8: np.int64(102600), 9: np.int64(308549), 10: np.int64(376796)}


In [4]:
# --- decoder + losses (from v2) + aux stem -----------------------------------
class PPM(nn.Module):
    def __init__(s,inc,outc,scales=(1,2,3,6)):
        super().__init__()
        s.st=nn.ModuleList([nn.Sequential(nn.AdaptiveAvgPool2d(sc),nn.Conv2d(inc,outc,1),nn.GroupNorm(8,outc),nn.GELU()) for sc in scales])
        s.out=nn.Sequential(nn.Conv2d(inc+len(scales)*outc,outc,3,padding=1),nn.GroupNorm(8,outc),nn.GELU())
    def forward(s,x):
        h,w=x.shape[-2:]; o=[x]
        for st in s.st: o.append(F.interpolate(st(x),(h,w),mode="bilinear",align_corners=False))
        return s.out(torch.cat(o,1))
class UPerHead(nn.Module):
    def __init__(s,dims,k,ch=256):
        super().__init__(); s.L=len(dims)
        s.lat=nn.ModuleList([nn.Conv2d(d,ch,1) for d in dims]); s.ppm=PPM(ch,ch)
        s.fpn=nn.ModuleList([nn.Sequential(nn.Conv2d(ch,ch,3,padding=1),nn.GroupNorm(8,ch),nn.GELU()) for _ in range(s.L)])
        s.fuse=nn.Sequential(nn.Conv2d(s.L*ch,ch,3,padding=1),nn.GroupNorm(8,ch),nn.GELU()); s.cls=nn.Conv2d(ch,k,1)
        s.scales=([4,2,1,0.5,0.25]+[0.25]*8)[:s.L]            # supports up to 5+ levels (aux = 5th)
    def forward(s,feats,out_hw=224):
        base=feats[0].shape[-2]; lats=[]
        for i,f in enumerate(feats):
            th=max(1,int(round(base*s.scales[i]))); lats.append(F.interpolate(s.lat[i](f),(th,th),mode="bilinear",align_corners=False))
        lats[-1]=s.ppm(lats[-1])
        for i in range(s.L-2,-1,-1): lats[i]=lats[i]+F.interpolate(lats[i+1],lats[i].shape[-2:],mode="bilinear",align_corners=False)
        lats=[s.fpn[i](lats[i]) for i in range(s.L)]; tgt=lats[0].shape[-2:]
        x=s.fuse(torch.cat([F.interpolate(l,tgt,mode="bilinear",align_corners=False) for l in lats],1))
        return F.interpolate(s.cls(x),(out_hw,out_hw),mode="bilinear",align_corners=False)
class AuxStem(nn.Module):                                     # 11 context bands 224->14
    def __init__(s,inc,outc=128):
        super().__init__(); s.net=nn.Sequential(
            nn.Conv2d(inc,64,3,2,1),nn.GroupNorm(8,64),nn.GELU(),
            nn.Conv2d(64,outc,3,2,1),nn.GroupNorm(8,outc),nn.GELU(),
            nn.Conv2d(outc,outc,3,2,1),nn.GroupNorm(8,outc),nn.GELU(),
            nn.Conv2d(outc,outc,3,2,1),nn.GroupNorm(8,outc),nn.GELU()); s.outc=outc
    def forward(s,x): return s.net(x)
def _lg(g):
    p=len(g); gts=g.sum(); inter=gts-g.float().cumsum(0); union=gts+(1-g).float().cumsum(0); jac=1.-inter/union
    if p>1: jac[1:p]=jac[1:p]-jac[0:-1]
    return jac
def lovasz(logits,labels,ig=255):
    pr=logits.softmax(1); B,C,H,W=pr.shape; pr=pr.permute(0,2,3,1).reshape(-1,C); lb=labels.reshape(-1); m=lb!=ig
    pr,lb=pr[m],lb[m]
    if lb.numel()==0: return logits.sum()*0.
    L=[]
    for c in range(C):
        fg=(lb==c).float()
        if fg.sum()==0: continue
        err=(fg-pr[:,c]).abs(); es,pm=torch.sort(err,0,descending=True); L.append(torch.dot(es,_lg(fg[pm])))
    return torch.stack(L).mean() if L else logits.sum()*0.
def dice(lg,t,ig=255,eps=1.):
    m=(t!=ig); tt=t.clone(); tt[~m]=0; oh=F.one_hot(tt,lg.shape[1]).permute(0,3,1,2).float()*m.unsqueeze(1)
    p=lg.softmax(1)*m.unsqueeze(1); i=(p*oh).sum((0,2,3)); u=p.sum((0,2,3))+oh.sum((0,2,3)); return 1-((2*i+eps)/(u+eps)).mean()
def soft_ce(logits,soft,valid):                               # DW soft-label distillation, masked
    lp=F.log_softmax(logits,1); has=(soft.sum(1)>0.5)&valid
    if has.sum()==0: return logits.sum()*0.
    return -(soft*lp).sum(1)[has].mean()


In [5]:
# --- encoder (Prithvi/ViT multi-level via hooks, from v2) ---------------------
def _find_blocks(enc):
    best=None
    for _,m in enc.named_modules():
        if isinstance(m,nn.ModuleList) and len(m)>=8 and (best is None or len(m)>len(best)): best=m
    return best
def _to_map(t):
    if t.dim()==4: return t
    B,N,D=t.shape; hw=int(round(N**.5))
    if hw*hw!=N: t=t[:,1:,:]; N=t.shape[1]; hw=int(round(N**.5))
    return t.transpose(1,2).reshape(B,D,hw,hw)
def make_encoder(name="prithvi_eo_v2_300"):
    from terratorch import BACKBONE_REGISTRY
    enc=BACKBONE_REGISTRY.build(name,pretrained=True,num_frames=1,
        bands=["BLUE","GREEN","RED","NIR_NARROW","SWIR_1","SWIR_2"]).to(dev)
    def run(x):
        try: return enc(x)
        except Exception: return enc(x.unsqueeze(2))
    blocks=_find_blocks(enc); cap={}
    idx=[int(round(f*(len(blocks)-1))) for f in (0.25,0.5,0.75,1.0)]
    def mk(j):
        def h(m,i,o): cap[j]=o
        return h
    for j,bi in enumerate(idx): blocks[bi].register_forward_hook(mk(j))
    def feats_fn(x):
        cap.clear(); _=run(x); return [_to_map(cap[j]) for j in range(4)]
    enc.eval()
    with torch.no_grad(): dims=[m.shape[1] for m in feats_fn(torch.zeros(1,len(PRITHVI_IDX),224,224,device=dev))]
    return enc, feats_fn, dims

class V3(nn.Module):
    def __init__(s,enc,feats_fn,dims):
        super().__init__(); s.enc,s.feats_fn=enc,feats_fn
        s.aux=AuxStem(len(AUX_IDX)) if USE_AUXSTEM else None
        s.head=UPerHead(dims+([s.aux.outc] if USE_AUXSTEM else []),K)
    def forward(s,x,ft):
        pin=x[:,PRITHVI_IDX]
        f=s.feats_fn(pin) if ft else [t.detach() for t in s.feats_fn(pin)]
        if USE_AUXSTEM: f=f+[s.aux(x[:,AUX_IDX])]
        return s.head(f)

def build_optim(enc,model,mode,base=1.5e-4,dcy=0.85):
    head_aux=[p for n,p in model.named_parameters() if not n.startswith("enc.")]
    if mode=="frozen":
        for p in enc.parameters(): p.requires_grad=False
        return torch.optim.AdamW([p for p in head_aux if p.requires_grad],lr=1e-3,weight_decay=1e-4)
    blocks=_find_blocks(enc); nb=len(blocks); groups=[{"params":head_aux,"lr":1e-3}]; seen=set()
    for i,blk in enumerate(blocks):
        ps=[p for p in blk.parameters() if p.requires_grad]
        for p in ps: seen.add(id(p))
        groups.append({"params":ps,"lr":base*(dcy**(nb-1-i))})
    rest=[p for p in enc.parameters() if p.requires_grad and id(p) not in seen]
    if rest: groups.append({"params":rest,"lr":base*(dcy**nb)})
    return torch.optim.AdamW(groups,weight_decay=1e-4)

@torch.no_grad()
def tta(fwd,x):
    ts=[lambda a:a, lambda a:torch.flip(a,[3]), lambda a:torch.flip(a,[2]), lambda a:torch.flip(a,[2,3])]
    return sum(t(fwd(t(x))).softmax(1) for t in ts)/len(ts)


In [ ]:
# --- train v3 ----------------------------------------------------------------
def run_v3(name="prithvi_eo_v2_300",seed=SEED):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    enc,feats_fn,dims=make_encoder(name); model=V3(enc,feats_fn,dims).to(dev)
    ft=(MODE=="finetune"); opt=build_optim(enc,model,MODE); bs=4 if ft else 8; accum=8//bs
    warm=torch.optim.lr_scheduler.LinearLR(opt,0.1,1.0,total_iters=WARMUP)
    cos=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS-WARMUP)
    sch=torch.optim.lr_scheduler.SequentialLR(opt,[warm,cos],milestones=[WARMUP])
    sc=torch.amp.GradScaler("cuda",enabled=(dev=="cuda")); ce=nn.CrossEntropyLoss(weight=CW.to(dev),ignore_index=255)
    smp=make_sampler()
    trdl=torch.utils.data.DataLoader(DS(tr,True),batch_size=bs,sampler=smp,shuffle=(smp is None),num_workers=2,drop_last=True)
    vadl=torch.utils.data.DataLoader(DS(va),batch_size=8,num_workers=2)
    for ep in range(EPOCHS):
        if ft: model.train()
        else:
            model.eval(); model.head.train()
            if USE_AUXSTEM: model.aux.train()
        opt.zero_grad()
        for j,(x,soft,y) in enumerate(trdl):
            x,soft,y=x.to(dev),soft.to(dev),y.to(dev)
            with torch.amp.autocast("cuda",enabled=(dev=="cuda")):
                lg=model(x,ft); loss=ce(lg,y)+dice(lg,y)+lovasz(lg,y)
                if USE_SOFT: loss=loss+LAM_SOFT*soft_ce(lg,soft,y!=255)
                loss=loss/accum
            sc.scale(loss).backward()
            if (j+1)%accum==0: sc.step(opt); sc.update(); opt.zero_grad()
        sch.step()
    model.eval(); P,T=[],[]
    with torch.no_grad():
        for x,soft,y in vadl:
            prob=tta(lambda a: model(a,False), x.to(dev))
            p=prob.argmax(1).cpu().numpy().ravel(); t=y.numpy().ravel(); k=t!=255; P.append(p[k]); T.append(t[k])
    P,T=np.concatenate(P),np.concatenate(T)
    iou=jaccard_score(T,P,average=None,zero_division=0,labels=list(range(K)))
    return dict(oa=accuracy_score(T,P),mf1=f1_score(T,P,average="macro",zero_division=0),
                miou=jaccard_score(T,P,average="macro",zero_division=0),per_class_iou=[round(float(v),3) for v in iou])

t0=time.time(); res={}
def _fmt(s): return f"{s/60:.1f}m" if s>=60 else f"{s:.0f}s"
# each config toggles ONE improvisation off vs 'full'; 'base' turns them ALL off.
CONFIGS = {
    "full":       dict(USE_AUXSTEM=1, USE_CONF=1, USE_SOFT=1, USE_COPYPASTE=1, USE_BALANCED=1),
    "-auxstem":   dict(USE_AUXSTEM=0, USE_CONF=1, USE_SOFT=1, USE_COPYPASTE=1, USE_BALANCED=1),
    "-conf":      dict(USE_AUXSTEM=1, USE_CONF=0, USE_SOFT=1, USE_COPYPASTE=1, USE_BALANCED=1),
    "-soft":      dict(USE_AUXSTEM=1, USE_CONF=1, USE_SOFT=0, USE_COPYPASTE=1, USE_BALANCED=1),
    "-copypaste": dict(USE_AUXSTEM=1, USE_CONF=1, USE_SOFT=1, USE_COPYPASTE=0, USE_BALANCED=1),
    "-balanced":  dict(USE_AUXSTEM=1, USE_CONF=1, USE_SOFT=1, USE_COPYPASTE=1, USE_BALANCED=0),
    "base(none)": dict(USE_AUXSTEM=0, USE_CONF=0, USE_SOFT=0, USE_COPYPASTE=0, USE_BALANCED=0),
}
CLS=["tree","shrub","grass","crop","built","bare","snow","water","wetland","mangrove","moss"]
for cname,cfg in CONFIGS.items():
    for k,v in cfg.items(): globals()[k]=bool(v)          # flags are module globals read by DS/V3
    ts=time.time(); r=run_v3(); res[cname]=r
    print(f"  {cname:<12} OA {r['oa']:.3f}  macroF1 {r['mf1']:.3f}  mIoU {r['miou']:.3f}   [{_fmt(time.time()-ts)}]", flush=True)
    json.dump(res, open("seg_v3_ablation.json","w"), indent=2)   # checkpoint after each config

full=res["full"]["mf1"]; rare=["snow","wetland","mangrove","moss"]; ri=[CLS.index(x) for x in rare]
print()
print(f"=== Prithvi-Seg v3 ABLATION on Dataset v2 ===  [total {_fmt(time.time()-t0)}]")
print(f"  {'config':<12} {'OA':>6} {'macroF1':>8} {'mIoU':>6}  {'dMF1':>6}  rare-mIoU(snow/wet/mang/moss)")
for cname,r in res.items():
    rr="/".join(f"{r['per_class_iou'][i]:.2f}" for i in ri)
    print(f"  {cname:<12} {r['oa']:.3f}  {r['mf1']:.3f}  {r['miou']:.3f}  {r['mf1']-full:+.3f}  {rr}")
print()
print("  read: '-X' row = drop improvisation X; dMF1 negative => X HELPS by that much.")
print("        'base(none)' = plain Prithvi+UPerNet on dsv2; full-minus-base = total gain.")


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Prithvi_EO_V2_300M.pt:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

  full         OA 0.871  macroF1 0.735  mIoU 0.617   [26.9m]


## Reading it
- **Baseline to beat:** v2 model on the old T1 chips = **0.663 macro-F1**. v3 here adds the
  richer bands + soft labels + conf-masking + copy-paste on the dsv2 data.
- **Ablate:** flip `USE_AUXSTEM / USE_CONF / USE_SOFT / USE_COPYPASTE / USE_BALANCED` to
  isolate each improvisation's contribution (rerun; ~2–3 h each).
- **Watch the rare classes** in per-class IoU (wetland/mangrove/snow/moss) — that's what
  copy-paste + targeted dsv2 chips are meant to lift.
- NOTE: dsv2 uses a zone-stratified (non-spatial) split, so numbers are NOT directly
  comparable to the bench_v1 spatial split — compare v3-vs-v3 ablations, not v3-vs-v2.
